# 10. Monte Carlo Uncertainty Analysis
## PCSI Uncertainty Propagation

This notebook reproduces Figure 10 from the CARBONICA paper: Monte Carlo uncertainty analysis for PCSI.

In [ ]:
# Import CARBONICA
import sys
sys.path.append('..')
from carbonica import CARBONICA
from carbonica.statistics.monte_carlo import MonteCarloPropagator
from carbonica.pcsi import PCSI

# Initialize
carbonica = CARBONICA(data_dir="../data")
pcsi_calc = PCSI()
mc = MonteCarloPropagator(n_ensemble=10000, seed=42)

In [ ]:
# Define parameters with uncertainties
params_2025 = carbonica.compute_current_state(2025)

params_with_uncertainty = {
    'NPP': (params_2025['NPP'], 4.2),
    'S_ocean': (params_2025['S_ocean'], 0.15),
    'G_atm': (params_2025['G_atm'], 0.05),
    'F_perma': (params_2025['F_perma'], 0.4),
    'beta': (params_2025['beta'], 0.005),
    'tau_soil': (params_2025['tau_soil'], 2.0),
    'E_anth': (params_2025['E_anth'], 0.3),
    'Phi_q': (params_2025['Phi_q'], 0.002)
}

print("Parameters with uncertainties (mean ± std):")
for param, (mean, std) in params_with_uncertainty.items():
    print(f"  {param:8s}: {mean:6.2f} ± {std:.2f}")

In [ ]:
# Run Monte Carlo propagation
print("\nRunning Monte Carlo propagation with 10,000 ensemble members...")
result = mc.propagate(params_with_uncertainty, pcsi_calc.compute)

print(f"\n=== PCSI Uncertainty Results ===")
print(f"Mean    : {result['mean']:.3f}")
print(f"Median  : {result['median']:.3f}")
print(f"Std Dev : {result['std']:.3f}")
print(f"5%      : {result['p5']:.3f}")
print(f"95%     : {result['p95']:.3f}")
print(f"Min     : {result['min']:.3f}")
print(f"Max     : {result['max']:.3f}")
print(f"Ensemble: {result['n_ensemble']} members")

In [ ]:
# Compare with paper values
print("\n=== Comparison with Paper ===")
paper_range = (0.73, 0.83)  # 5th-95th percentile from paper
print(f"Paper 5-95% range: {paper_range[0]:.2f} - {paper_range[1]:.2f}")
print(f"Our 5-95% range  : {result['p5']:.2f} - {result['p95']:.2f}")

if paper_range[0] <= result['p5'] and result['p95'] <= paper_range[1]:
    print("✅ Within paper range")
else:
    print("❌ Outside paper range")

In [ ]:
# Parameter sensitivity (simplified)
print("\n=== Parameter Sensitivity ===")
print("Varying one parameter at a time:")

base_params = params_2025.copy()
base_pcsi = pcsi_calc.compute(base_params)

sensitive_params = ['G_atm', 'F_perma', 'NPP']
for param in sensitive_params:
    test_params = base_params.copy()
    test_params[param] = base_params[param] * 1.1  # +10%
    new_pcsi = pcsi_calc.compute(test_params)
    change = (new_pcsi - base_pcsi) / base_pcsi * 100
    print(f"  {param:8s} +10% → PCSI change: {change:+.2f}%")

In [ ]:
# Projection with uncertainty
print("\n=== Projection with Uncertainty ===")

def simple_projection(params, years_from_now):
    # Simple linear projection
    proj = params.copy()
    proj['G_atm'] += 0.05 * years_from_now
    proj['F_perma'] += 0.12 * years_from_now
    return pcsi_calc.compute(proj)

print("Year  PCSI (mean ± std)")
print("-" * 30)

for year in [2030, 2040, 2050]:
    years_from_now = year - 2025
    ensemble = []
    for _ in range(1000):
        # Sample parameters
        sample = {}
        for param, (mean, std) in params_with_uncertainty.items():
            sample[param] = mc._normal_random(mean, std)
        proj_pcsi = simple_projection(sample, years_from_now)
        ensemble.append(proj_pcsi)
    
    mean = sum(ensemble) / len(ensemble)
    std = (sum((x - mean)**2 for x in ensemble) / (len(ensemble)-1))**0.5
    print(f"{year}: {mean:.3f} ± {std:.3f}")

In [ ]:
# Summary statistics
print("\n=== Summary ===")
print(f"PCSI 2025: {base_pcsi:.3f} (deterministic)")
print(f"PCSI 2025: {result['mean']:.3f} ± {result['std']:.3f} (Monte Carlo)")
print(f"Uncertainty range: ±{result['std']*2:.3f} (2σ)")